# Capitolo 5 — Algoritmi numerici (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap05_algoritmi_numerici_java.ipynb)

Notebook eseguibile del capitolo «Algoritmi numerici»: contiene gli algoritmi `ChangeOfBase`, `BasicPrimalityTest`, `PrimalityTest`, `SieveOfEratosthenes` e `EuclideanAlgorithm` scritti in Java ed eseguiti sugli stessi esempi della dispensa, con la stampa dei risultati e, dove la dispensa mostra una traccia, la traccia passo passo nello stesso formato.

Il capitolo completo — pseudocodice, esempi, dimostrazioni — è [sul sito](https://fabiofurini.github.io/algoritmi/algoritmi-di-base/05-algoritmi-numerici/). Lo stesso notebook è disponibile anche in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap05_algoritmi_numerici_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap05_algoritmi_numerici_cpp.ipynb).

**Come si esegue.** Su Colab basta cliccare sul badge e poi su *Runtime → Esegui tutto*, oppure eseguire le celle una alla volta con `Maiusc+Invio`. La prima cella verifica che il JDK sia presente e, se manca, lo installa. Ogni programma Java è scritto in un file con la cella `%%writefile Nome.java`, compilato con `javac` ed eseguito con `java` nella cella successiva. I programmi accettano l'input dalla riga di comando: la cella «Prova tu» lancia lo stesso programma con dei numeri da cambiare.

**Indici.** Nello pseudocodice gli array partono da 1, qui da 0: l'elemento $a[i]$ dello pseudocodice è `a[i-1]` nel codice. Le tracce stampate usano gli indici del codice (come negli esempi C++ della dispensa), quindi sono quelle dello pseudocodice con tutti gli indici diminuiti di 1. I nomi dei metodi sono quelli dello pseudocodice, nella convenzione di Java (`camelCase`: `findMaximum` per `FindMaximum`).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!java -version || apt-get -qq install -y default-jdk-headless

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


## ChangeOfBase

**Input:** un numero naturale $n \ge 1$ (in base 10) e una base $b \ge 2$. **Output:** l'array $d[1..nd]$ delle cifre di $n$ in base $b$ e il numero di cifre $nd$.

Una divisione per $b$ estrae la cifra meno significativa ($r = n \bmod b$) e lascia come quoziente il numero scritto con le altre cifre: si ripete finché il quoziente è 0. Le cifre escono dalla meno significativa alla più significativa, quindi nell'array `d` la cifra di peso $b^0$ sta in `d[0]` (in $d[1]$ nello pseudocodice).

Il numero di cifre $nd = \lfloor \log_b n \rfloor + 1$ è calcolato qui con le sole moltiplicazioni intere, senza logaritmi in virgola mobile: in doppia precisione `log(1000)/log(10)` vale $2.9999999999999996$ e la formula darebbe 3 cifre invece di 4.

In [2]:
%%writefile ChangeOfBase.java
import java.util.Arrays;

public class ChangeOfBase {

    // Restituisce le cifre di n in base b: d[0] è la cifra meno significativa, d.length è nd
    static int[] changeOfBase(int n, int b, boolean trace) {
        int nd = 1;                     // nd = ⌊log_b n⌋ + 1, con sole operazioni intere
        for (long p = b; p <= n; p *= b)
            nd = nd + 1;
        int[] d = new int[nd];          // array delle cifre
        int k = 0;                      // k = 1 nello pseudocodice
        while (n > 0) {
            int q = n / b;              // quoziente
            int r = n % b;              // resto (cifra in base b)
            d[k] = r;
            n = q;
            k = k + 1;
            if (trace)
                System.out.printf("%10d %5d %3d %5d %3d  %s\n", k, q, r, n, k, Arrays.toString(Arrays.copyOf(d, k)));
        }
        return d;
    }

    static String digitString(int[] d) {    // le cifre dalla più alta alla meno significativa
        StringBuilder s = new StringBuilder();
        for (int i = d.length - 1; i >= 0; i--) s.append("0123456789ABCDEF".charAt(d[i]));
        return s.toString();
    }

    static void run(int n, int b) {
        System.out.printf("Conversione: n = %d, b = %d\n", n, b);
        System.out.printf("%10s %5s %3s %5s %3s  %s\n", "iterazione", "q", "r", "n", "nd", "cifre");
        int[] d = changeOfBase(n, b, true);
        System.out.printf("Restituisce: nd = %d, d = %s\n", d.length, Arrays.toString(d));
        System.out.printf("%d = (%s)_%d\n\n", n, digitString(d), b);
    }

    public static void main(String[] args) {
        if (args.length > 1) {          // «Prova tu»: n e la base b dalla riga di comando
            run(Integer.parseInt(args[0]), Integer.parseInt(args[1]));
            return;
        }
        run(7, 2);
        run(8, 2);
        run(25, 3);
        run(100, 8);
        run(1000, 10);    // 4 cifre: il caso in cui log(1000)/log(10) in virgola mobile sbaglia
    }
}


Writing ChangeOfBase.java


### Esecuzione sugli esempi del capitolo

In [3]:
!javac ChangeOfBase.java && java ChangeOfBase

Conversione: n = 7, b = 2
iterazione     q   r     n  nd  cifre
         1     3   1     3   1  [1]
         2     1   1     1   2  [1, 1]
         3     0   1     0   3  [1, 1, 1]
Restituisce: nd = 3, d = [1, 1, 1]
7 = (111)_2

Conversione: n = 8, b = 2
iterazione     q   r     n  nd  cifre
         1     4   0     4   1  [0]
         2     2   0     2   2  [0, 0]
         3     1   0     1   3  [0, 0, 0]
         4     0   1     0   4  [0, 0, 0, 1]
Restituisce: nd = 4, d = [0, 0, 0, 1]
8 = (1000)_2

Conversione: n = 25, b = 3
iterazione     q   r     n  nd  cifre
         1     8   1     8   1  [1]
         2     2   2     2   2  [1, 2]
         3     0   2     0   3  [1, 2, 2]
Restituisce: nd = 3, d = [1, 2, 2]
25 = (221)_3

Conversione: n = 100, b = 8
iterazione     q   r     n  nd  cifre
         1    12   4    12   1  [4]
         2     1   4     1   2  [4, 4]
         3     0   1     0   3  [4, 4, 1]
Restituisce: nd = 3, d = [4, 4, 1]
100 = (144)_8

Conversione: n = 1000, b = 10

### Prova tu

Cambia i numeri qui sotto e riesegui la cella. I numeri sono $n$ e la base $b$ (fino a 16: le cifre oltre 9 si scrivono A, B, …).

In [4]:
!java ChangeOfBase 255 16

Conversione: n = 255, b = 16
iterazione     q   r     n  nd  cifre
         1    15  15    15   1  [15]
         2     0  15     0   2  [15, 15]
Restituisce: nd = 2, d = [15, 15]
255 = (FF)_16



## BasicPrimalityTest

**Input:** un numero naturale $n$. **Output:** `true` se $n$ è primo, `false` altrimenti.

Un divisore $d < n$ di $n$ è al più $n/2$: basta provare i candidati $d = 2, 3, \dots, \lfloor n/2 \rfloor$. Per un numero primo il ciclo viene eseguito per intero: per $n = 29$ i candidati sono 13. La funzione restituisce anche il numero di candidati provati.

In [5]:
%%writefile BasicPrimalityTest.java
public class BasicPrimalityTest {

    static int tested;                  // candidati provati dall'ultima chiamata

    // Restituisce true se n è primo
    static boolean basicPrimalityTest(int n, boolean trace) {
        tested = 0;
        if (n < 2) {
            if (trace)
                System.out.printf("%3s %8s  %s\n", "-", "-", "n < 2: restituisce false");
            return false;               // 0 e 1 non sono primi
        }
        for (int d = 2; d <= n / 2; d++) {  // d = 2..⌊n/2⌋
            tested++;
            if (n % d == 0) {
                if (trace)
                    System.out.printf("%3d %8d  %s\n", d, n % d, "restituisce false: d divide n");
                return false;           // d divide n: non è primo
            }
            if (trace)
                System.out.printf("%3d %8d  %s\n", d, n % d, "continua");
        }
        if (trace)
            System.out.printf("%3s %8s  %s\n", "-", "-", "il ciclo termina: restituisce true");
        return true;                    // nessun divisore: n è primo
    }

    static void run(int n) {
        System.out.printf("Controllo: n = %d, floor(n/2) = %d\n", n, n / 2);
        System.out.printf("%3s %8s  %s\n", "d", "n mod d", "azione");
        boolean result = basicPrimalityTest(n, true);
        System.out.printf("Restituisce: %s, candidati provati: %d\n\n", result ? "true" : "false", tested);
    }

    public static void main(String[] args) {
        if (args.length > 0) {          // «Prova tu»: i numeri da controllare arrivano dalla riga di comando
            for (String s : args) run(Integer.parseInt(s));
            return;
        }
        run(36);
        run(29);
        run(25);
    }
}


Writing BasicPrimalityTest.java


### Esecuzione sugli esempi del capitolo

In [6]:
!javac BasicPrimalityTest.java && java BasicPrimalityTest

Controllo: n = 36, floor(n/2) = 18
  d  n mod d  azione
  2        0  restituisce false: d divide n
Restituisce: false, candidati provati: 1

Controllo: n = 29, floor(n/2) = 14
  d  n mod d  azione
  2        1  continua
  3        2  continua
  4        1  continua
  5        4  continua
  6        5  continua
  7        1  continua
  8        5  continua
  9        2  continua
 10        9  continua
 11        7  continua
 12        5  continua
 13        3  continua
 14        1  continua
  -        -  il ciclo termina: restituisce true
Restituisce: true, candidati provati: 13

Controllo: n = 25, floor(n/2) = 12
  d  n mod d  azione
  2        1  continua
  3        1  continua
  4        1  continua
  5        0  restituisce false: d divide n
Restituisce: false, candidati provati: 4



### Prova tu

Cambia i numeri qui sotto e riesegui la cella. Il numero da controllare è $n$.

In [7]:
!java BasicPrimalityTest 97

Controllo: n = 97, floor(n/2) = 48
  d  n mod d  azione
  2        1  continua
  3        1  continua
  4        1  continua
  5        2  continua
  6        1  continua
  7        6  continua
  8        1  continua
  9        7  continua
 10        7  continua
 11        9  continua
 12        1  continua
 13        6  continua
 14       13  continua
 15        7  continua
 16        1  continua
 17       12  continua
 18        7  continua
 19        2  continua
 20       17  continua
 21       13  continua
 22        9  continua
 23        5  continua
 24        1  continua
 25       22  continua
 26       19  continua
 27       16  continua
 28       13  continua
 29       10  continua
 30        7  continua
 31        4  continua
 32        1  continua
 33       31  continua
 34       29  continua
 35       27  continua
 36       25  continua
 37       23  continua
 38       21  continua
 39       19  continua
 40       17  continua
 41       15  continua
 42       13  continua
 

## PrimalityTest

**Input:** un numero naturale $n$. **Output:** `true` se $n$ è primo, `false` altrimenti.

I divisori vengono in coppie $d$ e $n/d$ e in ogni coppia uno dei due è al più $\sqrt{n}$: un numero $n \ge 2$ è primo se e solo se nessun numero di $\{2, 3, \dots, \lfloor \sqrt{n} \rfloor\}$ lo divide. Per $n = 2$ e $n = 3$ il ciclo non viene eseguito e la funzione restituisce `true`. Per $n = 29$ i candidati sono 4, invece dei 13 dell'algoritmo di base.

In [8]:
%%writefile PrimalityTest.java
public class PrimalityTest {

    static int tested;                  // candidati provati dall'ultima chiamata

    // Restituisce true se n è primo
    static boolean primalityTest(int n, boolean trace) {
        tested = 0;
        if (n < 2) {
            if (trace)
                System.out.printf("%3s %8s  %s\n", "-", "-", "n < 2: restituisce false");
            return false;               // 0 e 1 non sono primi
        }
        for (int d = 2; d <= (int) Math.sqrt(n); d++) {     // d = 2..⌊√n⌋
            tested++;
            if (n % d == 0) {
                if (trace)
                    System.out.printf("%3d %8d  %s\n", d, n % d, "restituisce false: d divide n");
                return false;           // d divide n: non è primo
            }
            if (trace)
                System.out.printf("%3d %8d  %s\n", d, n % d, "continua");
        }
        if (trace)
            System.out.printf("%3s %8s  %s\n", "-", "-", "il ciclo termina: restituisce true");
        return true;                    // nessun divisore: n è primo
    }

    static void run(int n) {
        System.out.printf("Controllo: n = %d, floor(sqrt(n)) = %d\n", n, (int) Math.sqrt(n));
        System.out.printf("%3s %8s  %s\n", "d", "n mod d", "azione");
        boolean result = primalityTest(n, true);
        System.out.printf("Restituisce: %s, candidati provati: %d\n\n", result ? "true" : "false", tested);
    }

    public static void main(String[] args) {
        if (args.length > 0) {          // «Prova tu»: i numeri da controllare arrivano dalla riga di comando
            for (String s : args) run(Integer.parseInt(s));
            return;
        }
        run(36);
        run(29);
        run(25);
        run(2);         // il ciclo non viene eseguito
    }
}


Writing PrimalityTest.java


### Esecuzione sugli esempi del capitolo

In [9]:
!javac PrimalityTest.java && java PrimalityTest

Controllo: n = 36, floor(sqrt(n)) = 6
  d  n mod d  azione
  2        0  restituisce false: d divide n
Restituisce: false, candidati provati: 1

Controllo: n = 29, floor(sqrt(n)) = 5
  d  n mod d  azione
  2        1  continua
  3        2  continua
  4        1  continua
  5        4  continua
  -        -  il ciclo termina: restituisce true
Restituisce: true, candidati provati: 4

Controllo: n = 25, floor(sqrt(n)) = 5
  d  n mod d  azione
  2        1  continua
  3        1  continua
  4        1  continua
  5        0  restituisce false: d divide n
Restituisce: false, candidati provati: 4

Controllo: n = 2, floor(sqrt(n)) = 1
  d  n mod d  azione
  -        -  il ciclo termina: restituisce true
Restituisce: true, candidati provati: 0



### Prova tu

Cambia i numeri qui sotto e riesegui la cella. Il numero da controllare è $n$.

In [10]:
!java PrimalityTest 97

Controllo: n = 97, floor(sqrt(n)) = 9
  d  n mod d  azione
  2        1  continua
  3        1  continua
  4        1  continua
  5        2  continua
  6        1  continua
  7        6  continua
  8        1  continua
  9        7  continua
  -        -  il ciclo termina: restituisce true
Restituisce: true, candidati provati: 8



## SieveOfEratosthenes

**Input:** un numero naturale $n \ge 2$. **Output:** un array booleano $prime[1..n]$ in cui $prime[i] = 1$ se e solo se $i$ è primo.

Si parte con tutti i numeri segnati come primi (tranne 1) e, per ogni primo $p \le \lfloor \sqrt{n} \rfloor$, si segnano come composti i suoi multipli a partire da $p^2$. In Python, C++ e Java l'array parte da 0: `prime[i]` è associato al numero $i$, quindi ha $n+1$ elementi `prime[0..n]` e `prime[0]` (il numero 0, non primo) vale 0.

In [11]:
%%writefile SieveOfEratosthenes.java
import java.util.ArrayList;
import java.util.Arrays;
import java.util.List;

public class SieveOfEratosthenes {

    // Restituisce prime[0..n]: prime[i] = 1 se e solo se i è primo (n >= 2)
    static int[] sieveOfEratosthenes(int n, boolean trace) {
        int[] prime = new int[n + 1];
        Arrays.fill(prime, 1);          // prime[0..n], tutti a 1
        prime[0] = 0;                   // 0 non è primo
        prime[1] = 0;                   // 1 non è primo
        if (trace)
            System.out.printf("%5s  %s\n", "start", join(prime));

        for (int p = 2; p <= (int) Math.sqrt(n); p++) {     // p = 2..⌊√n⌋
            List<Integer> js = new ArrayList<>();           // i valori di j (solo per la traccia)
            if (prime[p] == 1) {
                // segna i multipli di p a partire da p^2
                for (int j = p * p; j <= n; j += p) {
                    prime[j] = 0;       // j non è primo
                    js.add(j);
                }
            }
            if (trace)
                System.out.printf("%5d  %s  %s\n", p, join(prime),
                        js.isEmpty() ? "nessun ciclo interno: prime[" + p + "] è 0"
                                     : "j = " + js.toString().replace("[", "").replace("]", ""));
        }
        return prime;
    }

    static String join(int[] a) {       // gli elementi separati da spazi (solo per la traccia)
        StringBuilder s = new StringBuilder();
        for (int i = 0; i < a.length; i++) s.append(i > 0 ? " " : "").append(a[i]);
        return s.toString();
    }

    static void run(int n) {
        System.out.printf("Crivello: n = %d, floor(sqrt(n)) = %d\n", n, (int) Math.sqrt(n));
        System.out.printf("%5s  prime[0..%d]\n", "p", n);
        int[] prime = sieveOfEratosthenes(n, true);
        List<Integer> primes = new ArrayList<>();
        for (int i = 0; i <= n; i++)
            if (prime[i] == 1) primes.add(i);
        System.out.printf("Numeri primi <= %d: %s\n\n", n, primes);
    }

    public static void main(String[] args) {
        if (args.length > 0) {          // «Prova tu»: n dalla riga di comando
            run(Integer.parseInt(args[0]));
            return;
        }
        run(10);
        run(20);
    }
}


Writing SieveOfEratosthenes.java


### Esecuzione sugli esempi del capitolo

In [12]:
!javac SieveOfEratosthenes.java && java SieveOfEratosthenes

Crivello: n = 10, floor(sqrt(n)) = 3
    p  prime[0..10]
start  0 0 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10
    3  0 0 1 1 0 1 0 1 0 0 0  j = 9
Numeri primi <= 10: [2, 3, 5, 7]

Crivello: n = 20, floor(sqrt(n)) = 4
    p  prime[0..20]
start  0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10, 12, 14, 16, 18, 20
    3  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0  j = 9, 12, 15, 18
    4  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0  nessun ciclo interno: prime[4] è 0
Numeri primi <= 20: [2, 3, 5, 7, 11, 13, 17, 19]



### Prova tu

Cambia i numeri qui sotto e riesegui la cella. Il numero è $n$ (almeno 2).

In [13]:
!java SieveOfEratosthenes 30

Crivello: n = 30, floor(sqrt(n)) = 5
    p  prime[0..30]
start  0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30
    3  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0  j = 9, 12, 15, 18, 21, 24, 27, 30
    4  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0  nessun ciclo interno: prime[4] è 0
    5  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 1 0  j = 25, 30
Numeri primi <= 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]



## EuclideanAlgorithm

**Input:** due numeri naturali $n, m \ge 1$. **Output:** $\gcd(n, m)$, il massimo comun divisore.

La coppia $(n, m)$ viene sostituita con $(m,\ n \bmod m)$, che ha gli stessi divisori comuni, finché il secondo numero è 0: allora $\gcd(n, 0) = n$. I parametri sono passati per valore: la funzione modifica le proprie copie. Due numeri sono coprimi se e solo se il risultato è 1.

In [14]:
%%writefile EuclideanAlgorithm.java
public class EuclideanAlgorithm {

    // Restituisce il massimo comun divisore di n e m (n, m >= 1)
    static int euclideanAlgorithm(int n, int m, boolean trace) {
        int step = 0;                   // solo per la traccia
        while (m != 0) {
            step++;
            int r = n % m;
            if (trace)
                System.out.printf("%5d %5d %5d  %d mod %d = %d\n", step, n, m, n, m, r);
            n = m;
            m = r;
        }
        if (trace)
            System.out.printf("%5d %5d %5d  %s\n", step + 1, n, m, "--- (il ciclo termina)");
        return n;                       // m = 0: n è il MCD
    }

    static void run(int n, int m) {
        System.out.printf("Calcolo del MCD: n = %d, m = %d\n", n, m);
        System.out.printf("%5s %5s %5s  %s\n", "passo", "n", "m", "r = n mod m");
        int g = euclideanAlgorithm(n, m, true);     // n e m restano invariati: passati per valore
        System.out.printf("Restituisce: gcd(%d, %d) = %d\n", n, m, g);
        System.out.printf("coprimi: %s\n\n", g == 1 ? "true" : "false");
    }

    public static void main(String[] args) {
        if (args.length > 1) {          // «Prova tu»: i due numeri dalla riga di comando
            run(Integer.parseInt(args[0]), Integer.parseInt(args[1]));
            return;
        }
        run(48, 18);
        run(252, 105);
        run(35, 12);
    }
}


Writing EuclideanAlgorithm.java


### Esecuzione sugli esempi del capitolo

In [15]:
!javac EuclideanAlgorithm.java && java EuclideanAlgorithm

Calcolo del MCD: n = 48, m = 18
passo     n     m  r = n mod m
    1    48    18  48 mod 18 = 12
    2    18    12  18 mod 12 = 6
    3    12     6  12 mod 6 = 0
    4     6     0  --- (il ciclo termina)
Restituisce: gcd(48, 18) = 6
coprimi: false

Calcolo del MCD: n = 252, m = 105
passo     n     m  r = n mod m
    1   252   105  252 mod 105 = 42
    2   105    42  105 mod 42 = 21
    3    42    21  42 mod 21 = 0
    4    21     0  --- (il ciclo termina)
Restituisce: gcd(252, 105) = 21
coprimi: false

Calcolo del MCD: n = 35, m = 12
passo     n     m  r = n mod m
    1    35    12  35 mod 12 = 11
    2    12    11  12 mod 11 = 1
    3    11     1  11 mod 1 = 0
    4     1     0  --- (il ciclo termina)
Restituisce: gcd(35, 12) = 1
coprimi: true



### Prova tu

Cambia i numeri qui sotto e riesegui la cella. I due numeri sono $n$ e $m$.

In [16]:
!java EuclideanAlgorithm 1071 462

Calcolo del MCD: n = 1071, m = 462
passo     n     m  r = n mod m
    1  1071   462  1071 mod 462 = 147
    2   462   147  462 mod 147 = 21
    3   147    21  147 mod 21 = 0
    4    21     0  --- (il ciclo termina)
Restituisce: gcd(1071, 462) = 21
coprimi: false



---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).